# Export the fine-tuned model to GGUF (8-bit) and check it still scores the same

CPU only. Downloads the merged model from the Hugging Face Hub, converts it with llama.cpp to Q8_0, scores it on 100 + 100 test examples next to the fp16 model, and uploads the GGUF for the live demo.

Needs the `HF_TOKEN` secret (write token). Internet on, no accelerator.

In [ ]:
import os
!rm -rf bank-txn-parser-qlora && git clone -q https://github.com/Tanzil-Ali-Rizvi/bank-txn-parser-qlora
%cd bank-txn-parser-qlora
os.environ["PYTHONPATH"] = "src"
!pip install -q llama-cpp-python==0.3.19 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt 2>&1 | tail -2

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import HfApi, snapshot_download
api = HfApi(token=os.environ["HF_TOKEN"])
repo = api.whoami()["name"] + "/bank-txn-parser-qwen2.5-1.5b"
model_dir = snapshot_download(repo, local_dir="/kaggle/working/model", token=os.environ["HF_TOKEN"])
!mkdir -p results/preds && cp /kaggle/working/model/results/preds/ft_*.jsonl results/preds/ && ls results/preds

In [ ]:
!mkdir -p gguf && python /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/working/model --outtype q8_0 --outfile gguf/bank-txn-parser-qwen2.5-1.5b-q8_0.gguf 2>&1 | tail -3
!ls -la gguf

In [ ]:
!python -m txnparser.gguf_eval --gguf gguf/bank-txn-parser-qwen2.5-1.5b-q8_0.gguf --n 100
!cat results/gguf_check.json

In [ ]:
api.upload_file(path_or_fileobj="gguf/bank-txn-parser-qwen2.5-1.5b-q8_0.gguf", path_in_repo="gguf/bank-txn-parser-qwen2.5-1.5b-q8_0.gguf", repo_id=repo, commit_message="Add Q8_0 GGUF for CPU inference")
api.upload_folder(folder_path="results", repo_id=repo, path_in_repo="results", allow_patterns=["gguf_check.json", "preds/gguf_*.jsonl"], commit_message="GGUF quality check")
!mkdir -p /kaggle/working/out && cp -r results/gguf_check.json results/preds/gguf_*.jsonl /kaggle/working/out/